# Step 6 — The Ingestion Pipeline (the schedulable \"data factory\")

Steps 2–5 built each stage as a reusable module. This step ties them into **one runnable, incremental pipeline** and shows how to **schedule** it.

```
fetch (Extract)  ->  clean + chunk + embed  ->  extract entities  ->  load into Neo4j
```

Two artifacts:
- **`signalpulse/pipeline.py`** — `run_pipeline(...)`, the orchestrator.
- **`run_pipeline.py`** — a command-line entry point you point a scheduler at.

The key property is **incrementality**: a scheduled run only spends LLM/embedding effort on documents that are *new or changed*, skipping everything it has already ingested.

## 1. How incrementality works

Every `RawDocument` carries a `content_hash` — a SHA-256 fingerprint of its text (created back in Step 2). Before doing any expensive work, the pipeline compares that hash to what's already in Neo4j:

| Situation | Meaning | Action | Status |
|---|---|---|---|
| no stored hash | brand-new document | process + load | `new` |
| stored hash **differs** | document changed | refresh chunks + reload | `updated` |
| stored hash **identical** | unchanged since last run | do nothing | `skipped` |

This is what makes the pipeline cheap to run on a schedule: the LLM is only invoked for genuinely new/changed content. (`force=True` overrides this and reprocesses everything.)

When a document is *updated*, its old chunks are deleted first (`delete_document_chunks`) so stale passages don't linger; shared `Entity` nodes are kept.

In [1]:
import os
import sys
from pathlib import Path

os.environ.setdefault("HF_HUB_DISABLE_SYMLINKS_WARNING", "1")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from signalpulse import connectors as C
from signalpulse import graph
from signalpulse.pipeline import run_pipeline

# Set True only when you want a fresh empty graph for this lesson.
# Leaving False keeps the denser demo corpus from `run_pipeline.py --profile demo`.
RESET_GRAPH = False

graph.verify_connectivity()
if RESET_GRAPH:
    graph.clear_data()
print("Neo4j ready. Starting counts:", graph.node_counts())


Neo4j ready. Starting counts: {'Document': 0, 'Chunk': 0, 'Entity': 0}


## 2. Run a small multi-source ingest (smoke-sized)

This cell keeps `limit=2` / `max_chunks_per_doc=2` so the notebook stays free-tier friendly while teaching the pipeline.

For the **richer company demo** (overlapping cyber + NIST + CMS, ~20 docs/source), use the CLI instead of wiping/rebuilding here:

```powershell
.\run_demo_ingest.ps1
# or:  python run_pipeline.py --profile demo
```

`demo_connectors()` + `INGEST_PROFILES` live in `signalpulse/connectors.py`.


In [2]:
# Smoke-sized multi-source ingest for this notebook.
print("Default sources:")
for c in C.default_connectors():
    print(f"  - {c.name:<22} [{c.domain}] ({c.tier})")

print("\nDemo sources (company / weekly profile):")
for c in C.demo_connectors():
    print(f"  - {c.name:<22} [{c.domain}] ({c.tier})")

report = run_pipeline(limit=2, max_chunks_per_doc=2)


Default sources:
  - cisa_kev               [Cybersecurity & Defense] (api)
  - nvd                    [Cybersecurity & Defense] (api)
  - fr_dod                 [Cybersecurity & Defense] (api)
  - fr_cms                 [Health IT & Civilian] (api)
  - fr_hhs_onc             [Health IT & Civilian] (api)
  - regulations_gov        [Health IT & Civilian] (api)
  - healthit_newsroom      [Health IT & Civilian] (scrape)
  - nist_news              [Tech Standards & Safety] (rss)
  - nist_csf               [Tech Standards & Safety] (scrape)
  - nist_80053_oscal       [Tech Standards & Safety] (api)
  - nist_rmf               [Tech Standards & Safety] (scrape)
  - nascio_priorities      [State & Local Gov] (seed)


[ok]   cisa_kev           2 docs


[ok]   nvd                2 docs


[ok]   fr_dod             2 docs


[ok]   fr_cms             2 docs


...

  [NEW ] nist_csf       https://nvlpubs.nist.gov/nistpubs/CSWP/N chunks=2


  [NEW ] nist_80053_oscal NIST-800-53-ac-1                         chunks=2


  [NEW ] nist_80053_oscal NIST-800-53-ac-2                         chunks=2


  [NEW ] nist_rmf       https://csrc.nist.gov/projects/risk-mana chunks=2


  [NEW ] nascio_priorities seed:nascio_priorities                   chunks=2

Done in 158.8s: {'new': 20, 'updated': 0, 'skipped': 0, 'failed': 0, 'chunks': 29, 'entity_mentions': 141, 'relationships': 53, 'seconds': 158.8}


## 3. Run it again → everything is skipped

This is the incremental payoff. Re-running immediately finds nothing changed, so it does **no** LLM work and finishes in a fraction of the time.

In [3]:
report2 = run_pipeline(limit=2, max_chunks_per_doc=2)

print("\nFirst run :", report.as_dict())
print("Second run:", report2.as_dict())
print(f"\nSecond run skipped everything: {report2.skipped > 0 and report2.new == 0}")
print(f"Second run was ~{report.seconds / max(report2.seconds, 0.1):.0f}x faster")

# Domain coverage in the graph
rows = graph.run_query("""
    MATCH (d:Document)
    RETURN d.domain AS domain, count(d) AS docs
    ORDER BY docs DESC
""")
print("\nDocuments by domain:")
for r in rows:
    print(f"  {r['domain']}: {r['docs']}")

[ok]   cisa_kev           2 docs


[ok]   nvd                2 docs


[ok]   fr_dod             2 docs


[ok]   fr_cms             2 docs


[ok]   fr_hhs_onc         2 docs


...

[ok]   healthit_newsroom  1 docs


[ok]   nist_news          2 docs


[ok]   nist_csf           1 docs


[ok]   nist_80053_oscal   2 docs


[ok]   nist_rmf           1 docs
[ok]   nascio_priorities  1 docs

Fetched 20 documents. Processing incrementally...

  [skip] cisa_kev       CVE-2026-56291                           chunks=0
  [skip] cisa_kev       CVE-2026-48939                           chunks=0
  [skip] nvd            CVE-2026-13523                           chunks=0
  [skip] nvd            CVE-2026-13524                           chunks=0
  [skip] fr_dod         2026-13931                               chunks=0
  [skip] fr_dod         2026-13914                               chunks=0
  [skip] fr_cms         2026-13918                               chunks=0
  [skip] fr_cms         2026-13865                               chunks=0
  [skip] fr_hhs_onc     2026-14073                               chunks=0
  [skip] fr_hhs_onc     2026-14063                               chunks=0
  [skip] regulations_gov EPA-HQ-OPPT-2025-0036-0002               chunks=0
  [skip] regulations_gov FDA-2026-N-2476-0001                     c

In [4]:
from signalpulse import loader as L

print("Graph after ingestion:")
for k, v in L.graph_summary().items():
    print(f"  {k:18s}: {v}")

Graph after ingestion:
  Document nodes    : 20
  Chunk nodes       : 29
  Entity nodes      : 122
  HAS_CHUNK rels    : 29
  MENTIONS rels     : 141
  RELATED_TO rels   : 53


## 4. Running it from the command line

The same pipeline runs headless via `run_pipeline.py` — this is what a scheduler calls:

```powershell
python run_pipeline.py --list-profiles
python run_pipeline.py --profile demo      # denser demo corpus (~20 docs/source)
python run_pipeline.py --profile weekly    # same depth; weekly refresh habit
.\run_demo_ingest.ps1                     # wrapper for demo
.\run_demo_ingest.ps1 -Weekly             # wrapper for weekly
python run_pipeline.py --profile full      # all default sources
python run_pipeline.py --profile smoke     # tiny plumbing check
python run_pipeline.py --source cisa_kev --limit 5
python run_pipeline.py --force             # reprocess even unchanged docs
```

Each successful run writes `data/processed/last_ingest.json`. Incremental re-runs skip unchanged docs (and will retry rate-limited failures once LLM quota resets).

## 5. Scheduling it

The pipeline is stateless between runs (state lives in Neo4j), so scheduling is just "run this command on a timer." Neo4j must be running first (`start_neo4j.ps1`).

**Windows — Task Scheduler** (weekly Monday 6 AM):

```powershell
# one-time setup (adjust the paths)
$py  = "C:\Users\saihaj\Documents\22nd Project\.venv\Scripts\python.exe"
$dir = "C:\Users\saihaj\Documents\22nd Project"
$action  = New-ScheduledTaskAction -Execute $py -Argument "run_pipeline.py --profile weekly" -WorkingDirectory $dir
$trigger = New-ScheduledTaskTrigger -Weekly -DaysOfWeek Monday -At 6am
Register-ScheduledTask -TaskName "SignalPulse Weekly Ingest" -Action $action -Trigger $trigger
```

**Linux / macOS — cron** (weekly Monday 6 AM):

```bash
# crontab -e , then add:
0 6 * * 1 cd /path/to/22nd\ Project && .venv/bin/python run_pipeline.py --profile weekly >> logs/pipeline.log 2>&1
```

**How often?** Weekly is a good habit for the demo profile; bump to daily if you want fresher CVEs. Because runs are incremental, extra runs are cheap (they mostly skip).

> The pipeline runs on a schedule and collects *new + changed* data; the chatbot agent (later steps) is separate and on-demand.


## Recap & what's next

We now have a complete, schedulable **ingestion pipeline**:

- **`run_pipeline()`** chains Extract → Clean → Chunk → Embed → Extract-entities → Load across chosen sources.
- **Profiles** (`demo` / `weekly` / `full` / `smoke`) pick source set + depth; see `INGEST_PROFILES` and `run_demo_ingest.ps1`.
- **Incremental**: new/updated/skipped by `content_hash`, so scheduled runs are cheap.
- **Resilient**: per-source and per-document failures are isolated; seed/fallback connectors cover blocked .gov sites.
- **`run_pipeline.py`** is the CLI a Task Scheduler / cron job runs; it stamps `last_ingest.json`.

The knowledge base now fills itself. The remaining steps build the **serving** side that answers questions from it.

**Next — Step 7:** the **retrieval** layer — vector (semantic) search over chunk embeddings, full-text keyword search, and graph lookups, each returning its source document + URL for citations.

> Prompt to continue:
> *"Step 7: Create notebook `07_retrieval.ipynb`. Implement the retrieval tools over Neo4j: vector search on chunk embeddings, full-text keyword search, and an entity/graph lookup. Each result should carry its source document and URL for citations. Show example queries and their retrieved, cited passages."*
